In [ ]:
# ==============================================================================
# CELDA 0: RUTAS DEL REPOSITORIO
# Este cuaderno leia sus datos desde Google Drive. Ahora los lee del propio
# repositorio, de modo que corre en cualquier clon sin configuracion previa.
# La raiz se resuelve buscando hacia arriba: funciona igual si el cuaderno se
# ejecuta desde su carpeta o desde la raiz del proyecto.
# ==============================================================================
from pathlib import Path

def _raiz_del_repositorio() -> Path:
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents]:
        if (carpeta / 'pyproject.toml').exists() and (carpeta / 'quanta').is_dir():
            return carpeta
    raise RuntimeError(
        'No se encontro la raiz del repositorio. Ejecute el cuaderno desde '
        'dentro del proyecto indice-sned.'
    )

RAIZ            = _raiz_del_repositorio()
RUTA_RAW        = (RAIZ / 'data' / 'raw').as_posix() + '/'
RUTA_PROCESADOS = (RAIZ / 'data' / 'processed').as_posix() + '/'
RUTA_REGISTRO   = (RAIZ / 'models' / 'registry').as_posix() + '/'
RUTA_METADATOS  = (RAIZ / 'models' / 'metadata').as_posix() + '/'

print('Raiz del repositorio:', RAIZ)


In [1]:
# ==============================================================================
# NOTEBOOK NUEVO: FORMATO_LARGO — CELDA 1
# Reconstrucción colegio × ciclo (corrige cluster desalineado + triplica N)
# ==============================================================================
import pandas as pd

RUTA = RUTA_PROCESADOS

# 1. Features (todas las fuentes integradas, con cluster/indicer/sel que hay que DESCARTAR)
df_v11 = pd.read_parquet(RUTA + 'tabla_modelo_final_v11.parquet')

# 2. SNED por ciclo (sin promediar) — fuente de cluster/indicer/sel correctos por ciclo
df_sned = pd.read_parquet(RUTA + 'sned_maestro_ciclos.parquet')

print("Columnas SNED disponibles:", [c for c in df_sned.columns if c in
      ['RBD','BIENIO_PREMIO','INDICER','SEL','CLUSTER','EFECTIVR','SUPERAR','INICIAR','MEJORAR','INTEGRAR','IGUALDR']])
print("\nCiclos en SNED:", df_sned['BIENIO_PREMIO'].unique())
print("Filas v11:", len(df_v11), "| columnas:", df_v11.shape[1])

Mounted at /content/drive
Columnas SNED disponibles: ['RBD', 'EFECTIVR', 'SUPERAR', 'INICIAR', 'MEJORAR', 'INTEGRAR', 'IGUALDR', 'CLUSTER', 'INDICER', 'SEL', 'BIENIO_PREMIO']

Ciclos en SNED: ['2016-17' '2018-19' '2020-21' '2022-23' '2024-25']
Filas v11: 7754 | columnas: 67


In [2]:
# ==============================================================================
# CELDA 2: CONSTRUCCIÓN DE LA TABLA EN FORMATO LARGO (colegio × ciclo)
# ==============================================================================
# Columnas de SNED que dependen del ciclo -> se traen POR CICLO, no promediadas
COLS_SNED_CICLO = ['EFECTIVR', 'SUPERAR', 'INICIAR', 'MEJORAR', 'INTEGRAR', 'IGUALDR',
                   'CLUSTER', 'INDICER', 'SEL']

# 1. De v11 quitamos las columnas SNED promediadas/de-un-ciclo (quedan solo features estables)
features_estables = df_v11.drop(columns=[c for c in COLS_SNED_CICLO if c in df_v11.columns])
print(f"Features estables (sin SNED por ciclo): {features_estables.shape[1]} columnas")

# 2. De SNED tomamos SOLO los 3 ciclos congelados (misma ventana SIMCE 2018-19 validada)
CICLOS_CONGELADOS = ['2020-21', '2022-23', '2024-25']
sned_largo = df_sned[df_sned['BIENIO_PREMIO'].isin(CICLOS_CONGELADOS)].copy()
sned_largo = sned_largo.rename(columns={'RBD': 'rbd'})
sned_largo = sned_largo[['rbd', 'BIENIO_PREMIO'] + COLS_SNED_CICLO]

# 3. Merge: cada colegio se expande a 3 filas (una por ciclo), con features estables repetidas
#    y cluster/indicer/sel CORRECTOS de cada ciclo
df_largo = pd.merge(sned_largo, features_estables, on='rbd', how='inner', validate='many_to_one')

print(f"\nTabla larga: {len(df_largo)} filas (esperado ~{len(df_v11)*3})")
print(f"Filas por ciclo:")
print(df_largo['BIENIO_PREMIO'].value_counts().sort_index())
print(f"Colegios únicos: {df_largo['rbd'].nunique()}")

# 4. Auditoría: ¿el cluster ahora varía por ciclo para el mismo colegio? (debe reflejar el 30,6%)
cluster_check = df_largo.groupby('rbd')['CLUSTER'].nunique()
print(f"\nColegios con cluster variable entre ciclos: {(cluster_check > 1).sum()} ({(cluster_check > 1).mean()*100:.1f}%)")

df_largo.to_parquet(RUTA + 'tabla_modelo_largo.parquet', index=False)
print(f"\nGuardado: tabla_modelo_largo.parquet | {df_largo.shape}")

Features estables (sin SNED por ciclo): 58 columnas

Tabla larga: 23111 filas (esperado ~23262)
Filas por ciclo:
BIENIO_PREMIO
2020-21    7752
2022-23    7710
2024-25    7649
Name: count, dtype: int64
Colegios únicos: 7754

Colegios con cluster variable entre ciclos: 2718 (35.1%)

Guardado: tabla_modelo_largo.parquet | (23111, 68)


In [3]:
# ==============================================================================
# CELDA 3: OPTIMIZACIÓN DE HIPERPARÁMETROS POR FACTOR (RandomizedSearchCV)
# ==============================================================================
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, GroupKFold, cross_val_predict
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import time

# Grilla escalada a 23.000 filas (Cap. 6 p.184 y Cap. 7 p.199)
GRILLA = {
    'n_estimators': [300, 500, 800],
    'max_depth': [5, 8, 10, 15, None],
    'min_samples_split': [20, 50, 100],
    'min_samples_leaf': [10, 20, 40],
    'max_features': ['sqrt', 'log2', 0.5],
    'bootstrap': [True],
}

def preparar_X(df_sub, features):
    """Imputación por mediana + indicador de ausencia (semántica MNAR)."""
    X = df_sub[features].copy()
    for c in features:
        if X[c].isnull().any():
            X[f'{c}_ausente'] = X[c].isnull().astype(int)
        X[c] = X[c].fillna(X[c].median())
    return X

# Actualizamos features de SUPERAR con las diferencias SIMCE (mejora validada: R² 0.02 -> 0.15)
FACTORES_SNED['SUPERAR']['features'] = cols_dif + cols_simce

resultados = {}
modelos_optimizados = {}
gkf = GroupKFold(n_splits=5)

for factor, cfg in FACTORES_SNED.items():
    t0 = time.time()
    feats = [f for f in cfg['features'] if f in df.columns]

    mask = df[factor].notna()
    if factor == 'SUPERAR':
        mask = mask & df['dif_simce_lect_4b'].notna()

    sub = df[mask]
    X = preparar_X(sub, feats)
    y = sub[factor]
    g = sub['rbd']

    busqueda = RandomizedSearchCV(
        RandomForestRegressor(random_state=42, n_jobs=-1),
        param_distributions=GRILLA,
        n_iter=25,
        cv=gkf.split(X, y, groups=g),
        scoring='r2',
        refit=True,
        random_state=42,
        n_jobs=-1,
    )
    busqueda.fit(X, y)

    # Evaluación out-of-fold con el mejor modelo
    pred = cross_val_predict(busqueda.best_estimator_, X, y,
                             cv=gkf, groups=g, n_jobs=-1)
    pred = np.clip(pred, 0, 100)  # acotar al rango válido del factor

    resultados[factor] = {
        'peso': cfg['peso'],
        'n': len(y),
        'r2': r2_score(y, pred),
        'mae': mean_absolute_error(y, pred),
        'rmse': np.sqrt(mean_squared_error(y, pred)),
        'best_params': busqueda.best_params_,
    }
    modelos_optimizados[factor] = busqueda.best_estimator_

    r = resultados[factor]
    print(f"{factor:<10} | peso {r['peso']:.2f} | n={r['n']:>5} | "
          f"R²={r['r2']:.3f} | MAE={r['mae']:5.2f} | RMSE={r['rmse']:5.2f} | {time.time()-t0:.0f}s")

print("\n--- Mejores hiperparámetros por factor ---")
for factor, r in resultados.items():
    print(f"\n{factor}: {r['best_params']}")

r2_pond = sum(r['r2'] * r['peso'] for r in resultados.values())
print(f"\nR² ponderado por peso SNED: {r2_pond:.3f}")

NameError: name 'cols_dif' is not defined